# Helpdesk loader — improved (henryk / U-ED-LSTM)

Sibling of `src/notebooks/loader_notebooks/normal/Helpdesk_full_loader.ipynb`.

- **§1 augmentation now happens in the training notebook** (oversampling Insert-ticket-start cases at the dataset level, train split only). This loader is back to a vanilla encode of the raw CSV — no synthetic events.
- **§3 Variant-index drop** — gated by `APPLY_S3_REMOVE_VARIANT` (default `False`). Flip to drop the post-hoc-leak column.

Output pickles land in the sibling `Loader/pkl/` folder.

In [1]:
import importlib
import sys
from pathlib import Path

import torch

sys.path.insert(0, '../../../../../../..')  # -> src/

import event_log_loader.new_event_log_loader
importlib.reload(event_log_loader.new_event_log_loader)
from event_log_loader.new_event_log_loader import EventLogLoader, EventLogDataset

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [2]:
# --- Config ---
PROJECT_ROOT = Path('../../../../../../../..').resolve()
SRC_CSV = PROJECT_ROOT / 'data' / 'helpdesk.csv'
OUT_DIR = Path('../pkl').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESULT_NAME = 'helpdesk_all'

# §3 — flip to True to drop 'Variant index' (post-hoc label leakage)
APPLY_S3_REMOVE_VARIANT = False

In [3]:
categorical_columns = [
    'Activity', 'Resource', 'Variant index',
    'seriousness', 'customer', 'product', 'responsible_section',
    'seriousness_2', 'service_level', 'service_type',
    'support_section', 'workgroup',
]

if APPLY_S3_REMOVE_VARIANT:
    categorical_columns = [c for c in categorical_columns if c != 'Variant index']
    print("§3 active: 'Variant index' removed")
else:
    print("§3 disabled: 'Variant index' kept (set APPLY_S3_REMOVE_VARIANT = True to drop it)")

event_log_properties = {
    'case_name': 'Case ID',
    'concept_name': 'Activity',
    'timestamp_name': 'Complete Timestamp',
    'date_format': '%Y/%m/%d %H:%M:%S.%f',
    'time_since_case_start_column': 'case_elapsed_time',
    'time_since_last_event_column': 'event_elapsed_time',
    'day_in_week_column': 'day_in_week',
    'seconds_in_day_column': 'seconds_in_day',
    'min_suffix_size': 5,
    'train_validation_size': 0.15,
    'test_validation_size': 0.2,
    'window_size': 'auto',
    'categorical_columns': categorical_columns,
    'continuous_columns': ['case_elapsed_time', 'event_elapsed_time', 'day_in_week', 'seconds_in_day'],
    'continuous_positive_columns': [],
}

event_log_loader = EventLogLoader(str(SRC_CSV), event_log_properties)
print('window_size:', event_log_loader.encoder_decoder.window_size)

§3 disabled: 'Variant index' kept (set APPLY_S3_REMOVE_VARIANT = True to drop it)
window_size: 18


In [4]:
train_dataset = event_log_loader.get_dataset('train')
out = OUT_DIR / f'{RESULT_NAME}_{event_log_loader.encoder_decoder.min_suffix_size}_train.pkl'
torch.save(train_dataset, out)
print(f'Saved {out}')

categorical tensors:   0%|          | 0/12 [00:00<?, ?it/s]

Activity:   0%|          | 0/2977 [00:00<?, ?it/s]

Resource:   0%|          | 0/2977 [00:00<?, ?it/s]

Variant index:   0%|          | 0/2977 [00:00<?, ?it/s]

seriousness:   0%|          | 0/2977 [00:00<?, ?it/s]

...

case_elapsed_time:   0%|          | 0/2977 [00:00<?, ?it/s]

event_elapsed_time:   0%|          | 0/2977 [00:00<?, ?it/s]

day_in_week:   0%|          | 0/2977 [00:00<?, ?it/s]

seconds_in_day:   0%|          | 0/2977 [00:00<?, ?it/s]

Saved /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/src/interpretability/improved_pipeline/henryk/helpdesk/improved/Loader/pkl/helpdesk_all_5_train.pkl


In [5]:
val_dataset = event_log_loader.get_dataset('val')
out = OUT_DIR / f'{RESULT_NAME}_{event_log_loader.encoder_decoder.min_suffix_size}_val.pkl'
torch.save(val_dataset, out)
print(f'Saved {out}')

categorical tensors:   0%|          | 0/12 [00:00<?, ?it/s]

Activity:   0%|          | 0/687 [00:00<?, ?it/s]

Resource:   0%|          | 0/687 [00:00<?, ?it/s]

Variant index:   0%|          | 0/687 [00:00<?, ?it/s]

seriousness:   0%|          | 0/687 [00:00<?, ?it/s]

...

case_elapsed_time:   0%|          | 0/687 [00:00<?, ?it/s]

event_elapsed_time:   0%|          | 0/687 [00:00<?, ?it/s]

day_in_week:   0%|          | 0/687 [00:00<?, ?it/s]

seconds_in_day:   0%|          | 0/687 [00:00<?, ?it/s]

Saved /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/src/interpretability/improved_pipeline/henryk/helpdesk/improved/Loader/pkl/helpdesk_all_5_val.pkl


In [6]:
test_dataset = event_log_loader.get_dataset('test')
out = OUT_DIR / f'{RESULT_NAME}_{event_log_loader.encoder_decoder.min_suffix_size}_test.pkl'
torch.save(test_dataset, out)
print(f'Saved {out}')

categorical tensors:   0%|          | 0/12 [00:00<?, ?it/s]

Activity:   0%|          | 0/916 [00:00<?, ?it/s]

Resource:   0%|          | 0/916 [00:00<?, ?it/s]

Variant index:   0%|          | 0/916 [00:00<?, ?it/s]

seriousness:   0%|          | 0/916 [00:00<?, ?it/s]

...

case_elapsed_time:   0%|          | 0/916 [00:00<?, ?it/s]

event_elapsed_time:   0%|          | 0/916 [00:00<?, ?it/s]

day_in_week:   0%|          | 0/916 [00:00<?, ?it/s]

seconds_in_day:   0%|          | 0/916 [00:00<?, ?it/s]

Saved /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/src/interpretability/improved_pipeline/henryk/helpdesk/improved/Loader/pkl/helpdesk_all_5_test.pkl
